# Sistema de Recomendacao de Livros

## IAA012 - Frameworks de IA
### Especializacao em Inteligencia Artificial Aplicada - UFPR/SEPT

---

Este notebook implementa um **Sistema de Recomendacao de Livros** utilizando **Filtragem Colaborativa** com **Redes Neurais** e **Embeddings**.

**Base de dados:** Base_livros.csv
- **ISBN:** Identificador unico do livro
- **Titulo:** Nome do livro
- **Autor:** Autor do livro
- **Ano:** Ano de publicacao
- **Editora:** Editora do livro
- **ID_usuario:** Identificacao do usuario
- **Notas:** Avaliacao dada pelo usuario ao livro (0-10)

## 1. Importacao das Bibliotecas

In [ ]:
# Instalacao de dependencias (se necessario no Colab)
# !pip install tensorflow pandas numpy matplotlib scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")

## 2. Carregamento e Exploracao dos Dados

In [ ]:
# Para Google Colab: fazer upload do arquivo ou montar o Google Drive
# from google.colab import files
# uploaded = files.upload()

# Carregar a base de dados
df = pd.read_csv('Base_livros.csv')

# Visualizar as primeiras linhas
print("Primeiras linhas do dataset:")
df.head(10)

In [ ]:
# Informacoes gerais do dataset
print("\nInformacoes do Dataset:")
print(f"Numero de registros: {len(df)}")
print(f"Numero de colunas: {len(df.columns)}")
print(f"\nColunas: {df.columns.tolist()}")
print(f"\nTipos de dados:")
print(df.dtypes)

In [ ]:
# Estatisticas descritivas
print("\nEstatisticas das Notas:")
print(df['Notas'].describe())

print(f"\nNumero de usuarios unicos: {df['ID_usuario'].nunique()}")
print(f"Numero de livros unicos: {df['ISBN'].nunique()}")
print(f"Numero de titulos unicos: {df['Titulo'].nunique()}")

In [ ]:
# Verificar valores ausentes
print("\nValores ausentes por coluna:")
print(df.isnull().sum())

In [ ]:
# Distribuicao das notas
plt.figure(figsize=(10, 5))
df['Notas'].hist(bins=11, edgecolor='black', alpha=0.7)
plt.xlabel('Notas')
plt.ylabel('Frequencia')
plt.title('Distribuicao das Notas dos Livros')
plt.xticks(range(0, 11))
plt.grid(axis='y', alpha=0.3)
plt.show()

## 3. Pre-processamento dos Dados

In [ ]:
# Pre-processamento dos dados
# Remover registros com valores ausentes (se houver)
df_clean = df.dropna(subset=['ID_usuario', 'ISBN', 'Notas']).copy()

# Converter ID_usuario para string para garantir consistencia
df_clean['ID_usuario'] = df_clean['ID_usuario'].astype(str)
df_clean['ISBN'] = df_clean['ISBN'].astype(str)

# Para datasets grandes, podemos filtrar usuarios e livros com poucas avaliacoes
# Neste caso, usaremos um limite baixo para manter mais dados
min_user_ratings = 2  # Usuarios com pelo menos 2 avaliacoes
min_book_ratings = 2  # Livros com pelo menos 2 avaliacoes

# Contar avaliacoes por usuario
user_counts = df_clean['ID_usuario'].value_counts()
valid_users = user_counts[user_counts >= min_user_ratings].index

# Filtrar apenas por usuarios validos primeiro
df_filtered = df_clean[df_clean['ID_usuario'].isin(valid_users)].copy()

# Contar avaliacoes por livro apos filtro de usuarios
book_counts = df_filtered['ISBN'].value_counts()
valid_books = book_counts[book_counts >= min_book_ratings].index

# Aplicar filtro de livros
df_filtered = df_filtered[df_filtered['ISBN'].isin(valid_books)].copy()

print(f"Dataset original: {len(df)} registros")
print(f"Dataset limpo: {len(df_clean)} registros")
print(f"Dataset filtrado: {len(df_filtered)} registros")
print(f"Usuarios validos: {len(df_filtered['ID_usuario'].unique())}")
print(f"Livros validos: {len(df_filtered['ISBN'].unique())}")

In [ ]:
# Criar encoders para usuarios e livros
user_encoder = LabelEncoder()
book_encoder = LabelEncoder()

df_filtered['user_encoded'] = user_encoder.fit_transform(df_filtered['ID_usuario'])
df_filtered['book_encoded'] = book_encoder.fit_transform(df_filtered['ISBN'])

# Numero de usuarios e livros unicos
n_users = df_filtered['user_encoded'].nunique()
n_books = df_filtered['book_encoded'].nunique()

print(f"Numero de usuarios (encoded): {n_users}")
print(f"Numero de livros (encoded): {n_books}")
print(f"Total de interacoes: {len(df_filtered)}")

In [ ]:
# Criar mapeamento de ISBN para Titulo
isbn_to_title = df_filtered.drop_duplicates('ISBN').set_index('ISBN')['Titulo'].to_dict()
encoded_to_isbn = dict(zip(df_filtered['book_encoded'], df_filtered['ISBN']))

# Visualizar alguns mapeamentos
print("Exemplo de mapeamentos ISBN -> Titulo:")
for i, (isbn, titulo) in enumerate(list(isbn_to_title.items())[:5]):
    print(f"  {isbn}: {titulo[:50]}..." if len(titulo) > 50 else f"  {isbn}: {titulo}")

In [ ]:
# Normalizar as notas para o intervalo [0, 1]
min_rating = df_filtered['Notas'].min()
max_rating = df_filtered['Notas'].max()

df_filtered['rating_normalized'] = (df_filtered['Notas'] - min_rating) / (max_rating - min_rating)

print(f"Notas originais: [{min_rating}, {max_rating}]")
print(f"Notas normalizadas: [{df_filtered['rating_normalized'].min():.2f}, {df_filtered['rating_normalized'].max():.2f}]")

In [ ]:
# Dividir em conjuntos de treino e teste
X = df_filtered[['user_encoded', 'book_encoded']].values
y = df_filtered['rating_normalized'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Conjunto de treino: {len(X_train)} amostras")
print(f"Conjunto de teste: {len(X_test)} amostras")

## 4. Construcao do Modelo de Recomendacao

Utilizaremos um modelo de **Filtragem Colaborativa** baseado em **Neural Collaborative Filtering (NCF)** com camadas de **Embedding** para representar usuarios e livros em um espaco vetorial latente.

In [ ]:
# Hiperparametros do modelo
EMBEDDING_SIZE = 50  # Dimensao dos embeddings
LEARNING_RATE = 0.001
EPOCHS = 25
BATCH_SIZE = 64

In [ ]:
def create_recommendation_model(n_users, n_books, embedding_size=50):
    """
    Cria um modelo de recomendacao usando Neural Collaborative Filtering.
    
    Arquitetura:
    - Embedding layer para usuarios
    - Embedding layer para livros
    - Concatenacao dos embeddings
    - Camadas densas para aprender interacoes nao-lineares
    - Saida: predicao da nota normalizada
    """
    
    # Input layers
    user_input = Input(shape=(1,), name='user_input')
    book_input = Input(shape=(1,), name='book_input')
    
    # Embedding layers
    user_embedding = Embedding(
        input_dim=n_users, 
        output_dim=embedding_size, 
        name='user_embedding'
    )(user_input)
    user_vec = Flatten(name='user_flatten')(user_embedding)
    
    book_embedding = Embedding(
        input_dim=n_books, 
        output_dim=embedding_size, 
        name='book_embedding'
    )(book_input)
    book_vec = Flatten(name='book_flatten')(book_embedding)
    
    # Concatenar embeddings
    concat = Concatenate(name='concat')([user_vec, book_vec])
    
    # Camadas densas
    dense1 = Dense(128, activation='relu', name='dense1')(concat)
    dropout1 = Dropout(0.3, name='dropout1')(dense1)
    
    dense2 = Dense(64, activation='relu', name='dense2')(dropout1)
    dropout2 = Dropout(0.2, name='dropout2')(dense2)
    
    dense3 = Dense(32, activation='relu', name='dense3')(dropout2)
    
    # Camada de saida
    output = Dense(1, activation='sigmoid', name='output')(dense3)
    
    # Criar modelo
    model = Model(inputs=[user_input, book_input], outputs=output)
    
    return model

In [ ]:
# Criar o modelo
model = create_recommendation_model(n_users, n_books, EMBEDDING_SIZE)

# Compilar o modelo
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='mse',  # Mean Squared Error para regressao
    metrics=['mae']  # Mean Absolute Error
)

# Resumo do modelo
model.summary()

In [ ]:
# Visualizar a arquitetura do modelo
try:
    from tensorflow.keras.utils import plot_model
    plot_model(model, show_shapes=True, show_layer_names=True, to_file='model_architecture.png')
    from IPython.display import Image
    display(Image('model_architecture.png'))
except:
    print("Nao foi possivel gerar a visualizacao do modelo (requer graphviz)")

## 5. Treinamento do Modelo

In [ ]:
# Callback para early stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Preparar dados de entrada
train_users = X_train[:, 0]
train_books = X_train[:, 1]
test_users = X_test[:, 0]
test_books = X_test[:, 1]

print(f"Iniciando treinamento com {EPOCHS} epochs...")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")

In [ ]:
# Treinar o modelo
history = model.fit(
    [train_users, train_books],
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=([test_users, test_books], y_test),
    callbacks=[early_stopping],
    verbose=1
)

## 6. Avaliacao do Modelo - Graficos de Loss

### Analise dos Graficos de Perda (Loss)

Os graficos de loss sao fundamentais para avaliar o desempenho e o comportamento do modelo durante o treinamento.

In [ ]:
# Plotar historico de treinamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafico de Loss
axes[0].plot(history.history['loss'], label='loss (treino)', linewidth=2)
axes[0].plot(history.history['val_loss'], label='val_loss (validacao)', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Funcao de Perda (Loss) durante o Treinamento')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Grafico de MAE
axes[1].plot(history.history['mae'], label='MAE (treino)', linewidth=2)
axes[1].plot(history.history['val_mae'], label='val_MAE (validacao)', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Erro Absoluto Medio (MAE) durante o Treinamento')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Grafico detalhado da Loss
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], 'b-', label='loss (treino)', linewidth=2, marker='o', markersize=4)
plt.plot(history.history['val_loss'], 'r-', label='val_loss (validacao)', linewidth=2, marker='s', markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Evolucao da Funcao de Perda (Loss)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Estatisticas finais
print("\n" + "="*60)
print("ESTATISTICAS FINAIS DO TREINAMENTO")
print("="*60)
print(f"Loss final (treino): {history.history['loss'][-1]:.4f}")
print(f"Loss final (validacao): {history.history['val_loss'][-1]:.4f}")
print(f"MAE final (treino): {history.history['mae'][-1]:.4f}")
print(f"MAE final (validacao): {history.history['val_mae'][-1]:.4f}")
print("="*60)

### Interpretacao dos Graficos de Loss

**O que os graficos nos dizem:**

1. **Convergencia do Modelo:**
   - A diminuicao consistente da loss ao longo das epochs indica que o modelo esta aprendendo padroes nos dados
   - Quanto mais rapida a queda inicial, mais eficiente e o aprendizado

2. **Comparacao Treino vs Validacao:**
   - **Curvas proximas:** Indica boa generalizacao - o modelo nao esta decorando os dados de treino
   - **Gap crescente:** Pode indicar overfitting - o modelo esta memorizando ao inves de generalizar
   - **Validacao melhor que treino:** Pode ocorrer devido ao Dropout (ativo apenas no treino)

3. **Estabilizacao:**
   - A estabilizacao da loss indica que o modelo atingiu sua capacidade maxima de aprendizado
   - Early stopping ajuda a parar no ponto otimo antes de overfitting

4. **Valores de Loss:**
   - MSE (Mean Squared Error) penaliza erros grandes mais fortemente
   - Valores baixos indicam que as predicoes estao proximas dos valores reais

## 7. Sistema de Recomendacao de Livros

In [ ]:
def get_user_recommendations(user_id, model, df_filtered, user_encoder, book_encoder, 
                             isbn_to_title, n_recommendations=10):
    """
    Gera recomendacoes de livros para um usuario especifico.
    
    Parametros:
    - user_id: ID original do usuario
    - model: modelo treinado
    - n_recommendations: numero de recomendacoes a retornar
    
    Retorna:
    - DataFrame com os livros recomendados e notas previstas
    """
    
    # Converter user_id para string (mesmo formato do encoder)
    user_id_str = str(user_id)
    
    # Verificar se o usuario existe
    if user_id_str not in user_encoder.classes_:
        print(f"Usuario {user_id} nao encontrado no dataset!")
        return None
    
    # Obter o ID encoded do usuario
    user_encoded = user_encoder.transform([user_id_str])[0]
    
    # Obter livros ja avaliados pelo usuario
    user_rated_books = df_filtered[df_filtered['ID_usuario'] == user_id_str]['book_encoded'].values
    
    # Obter todos os livros nao avaliados pelo usuario
    all_books = df_filtered['book_encoded'].unique()
    books_to_predict = np.array([b for b in all_books if b not in user_rated_books])
    
    if len(books_to_predict) == 0:
        print(f"Usuario {user_id} ja avaliou todos os livros disponiveis!")
        return None
    
    # Preparar dados para predicao
    user_array = np.array([user_encoded] * len(books_to_predict))
    
    # Fazer predicoes
    predictions = model.predict([user_array, books_to_predict], verbose=0).flatten()
    
    # Desnormalizar as predicoes para a escala original (0-10)
    predictions_original = predictions * (max_rating - min_rating) + min_rating
    
    # Criar DataFrame com resultados
    results = pd.DataFrame({
        'book_encoded': books_to_predict,
        'nota_prevista': predictions_original
    })
    
    # Ordenar por nota prevista (decrescente)
    results = results.sort_values('nota_prevista', ascending=False).head(n_recommendations)
    
    # Adicionar informacoes do livro
    results['ISBN'] = results['book_encoded'].map(encoded_to_isbn)
    results['Titulo'] = results['ISBN'].map(isbn_to_title)
    
    return results[['ISBN', 'Titulo', 'nota_prevista']]

In [ ]:
def show_user_history(user_id, df_filtered):
    """
    Mostra o historico de avaliacoes de um usuario.
    """
    user_id_str = str(user_id)
    user_history = df_filtered[df_filtered['ID_usuario'] == user_id_str][['Titulo', 'Autor', 'Notas']]
    user_history = user_history.sort_values('Notas', ascending=False)
    return user_history

## 8. Exemplo de Recomendacao para um Usuario

Vamos demonstrar o sistema de recomendacao selecionando um usuario e gerando recomendacoes personalizadas de livros.

In [ ]:
# Selecionar um usuario com um bom numero de avaliacoes para demonstracao
user_activity = df_filtered.groupby('ID_usuario').size().sort_values(ascending=False)
print("Top 10 usuarios mais ativos:")
print(user_activity.head(10))

# Selecionar o usuario para demonstracao
sample_user_id = user_activity.index[0]  # Usuario mais ativo
print(f"\nUsuario selecionado para demonstracao: {sample_user_id}")
print(f"Numero de avaliacoes deste usuario: {user_activity[sample_user_id]}")

In [ ]:
# Mostrar historico de avaliacoes do usuario
print(f"\n{'='*80}")
print(f"HISTORICO DE AVALIACOES DO USUARIO {sample_user_id}")
print(f"{'='*80}")
print("\nLivros com melhores notas dadas pelo usuario:")

user_history = show_user_history(sample_user_id, df_filtered)
display(user_history.head(10))

In [ ]:
# Gerar recomendacoes para o usuario
print(f"\n{'='*80}")
print(f"RECOMENDACOES DE LIVROS PARA O USUARIO {sample_user_id}")
print(f"{'='*80}")

recommendations = get_user_recommendations(
    user_id=sample_user_id,
    model=model,
    df_filtered=df_filtered,
    user_encoder=user_encoder,
    book_encoder=book_encoder,
    isbn_to_title=isbn_to_title,
    n_recommendations=10
)

print("\nTop 10 livros recomendados:")
display(recommendations)

In [ ]:
# Visualizacao das recomendacoes
plt.figure(figsize=(12, 6))
titles = [t[:30] + '...' if len(t) > 30 else t for t in recommendations['Titulo'].values]
notas = recommendations['nota_prevista'].values

bars = plt.barh(range(len(titles)), notas, color='steelblue', edgecolor='navy')
plt.yticks(range(len(titles)), titles)
plt.xlabel('Nota Prevista', fontsize=12)
plt.title(f'Top 10 Recomendacoes de Livros para Usuario {sample_user_id}', fontsize=14)
plt.xlim(0, 10)
plt.gca().invert_yaxis()

# Adicionar valores nas barras
for bar, nota in zip(bars, notas):
    plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
             f'{nota:.2f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### Interpretacao das Recomendacoes

**Como interpretar os resultados:**

1. **Nota Prevista:**
   - Representa a estimativa do modelo sobre quanto o usuario gostaria do livro (escala 0-10)
   - Notas mais altas indicam maior compatibilidade entre o usuario e o livro

2. **Base para Recomendacao:**
   - O modelo aprende padroes de preferencia a partir das avaliacoes existentes
   - Usuarios com gostos similares tendem a receber recomendacoes similares
   - Livros bem avaliados por usuarios similares serao recomendados

3. **Filtragem Colaborativa:**
   - Nao depende de caracteristicas explicitas dos livros (genero, autor)
   - Baseia-se puramente nos padroes de avaliacao dos usuarios
   - Pode descobrir conexoes nao obvias entre livros

In [ ]:
# Testar recomendacoes para outro usuario
print("\n" + "="*80)
print("RECOMENDACOES PARA OUTRO USUARIO")
print("="*80)

# Selecionar outro usuario
another_user_id = user_activity.index[5]  # 6o usuario mais ativo
print(f"\nUsuario: {another_user_id}")
print(f"Numero de avaliacoes: {user_activity[another_user_id]}")

# Historico
print("\nAlguns livros avaliados por este usuario:")
display(show_user_history(another_user_id, df_filtered).head(5))

# Recomendacoes
print("\nRecomendacoes para este usuario:")
recommendations_2 = get_user_recommendations(
    user_id=another_user_id,
    model=model,
    df_filtered=df_filtered,
    user_encoder=user_encoder,
    book_encoder=book_encoder,
    isbn_to_title=isbn_to_title,
    n_recommendations=5
)
display(recommendations_2)

## 9. Avaliacao Final do Modelo

In [ ]:
# Avaliar o modelo no conjunto de teste
print("Avaliacao final do modelo no conjunto de teste:")
test_loss, test_mae = model.evaluate([test_users, test_books], y_test, verbose=0)

# Converter MAE para escala original
test_mae_original = test_mae * (max_rating - min_rating)

print(f"\nLoss (MSE) no teste: {test_loss:.4f}")
print(f"MAE no teste (normalizado): {test_mae:.4f}")
print(f"MAE no teste (escala original 0-10): {test_mae_original:.4f}")
print(f"\nIsso significa que, em media, as predicoes do modelo erram por aproximadamente {test_mae_original:.2f} pontos na escala de 0-10.")

In [ ]:
# Comparar predicoes vs valores reais
predictions_test = model.predict([test_users, test_books], verbose=0).flatten()
predictions_test_original = predictions_test * (max_rating - min_rating) + min_rating
y_test_original = y_test * (max_rating - min_rating) + min_rating

plt.figure(figsize=(10, 6))
plt.scatter(y_test_original, predictions_test_original, alpha=0.3, s=10)
plt.plot([0, 10], [0, 10], 'r--', linewidth=2, label='Predicao perfeita')
plt.xlabel('Nota Real', fontsize=12)
plt.ylabel('Nota Prevista', fontsize=12)
plt.title('Predicoes vs Valores Reais', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(-0.5, 10.5)
plt.ylim(-0.5, 10.5)
plt.tight_layout()
plt.show()

## 10. Conclusao

### Resumo do Sistema de Recomendacao

Neste notebook, implementamos um **Sistema de Recomendacao de Livros** utilizando:

1. **Filtragem Colaborativa** baseada em Neural Collaborative Filtering (NCF)
2. **Embeddings** para representar usuarios e livros em espacos vetoriais latentes
3. **Rede Neural Profunda** para aprender interacoes nao-lineares entre usuarios e livros

### Principais Aprendizados

**Graficos de Loss:**
- A diminuicao da loss indica aprendizado efetivo
- Curvas de treino e validacao proximas indicam boa generalizacao
- Early stopping previne overfitting

**Sistema de Recomendacao:**
- Embeddings capturam preferencias latentes de usuarios e caracteristicas de livros
- O modelo pode prever notas para livros nao avaliados
- Recomendacoes sao personalizadas com base no historico de cada usuario

### Possiveis Melhorias

- Incorporar informacoes adicionais (autor, genero, ano)
- Utilizar TensorFlow Recommenders para abordagens mais avancadas
- Implementar sistemas hibridos (colaborativo + baseado em conteudo)
- Ajustar hiperparametros para melhorar performance

In [ ]:
print("Sistema de Recomendacao de Livros - Implementacao Concluida!")
print(f"\nResumo:")
print(f"- Total de usuarios: {n_users}")
print(f"- Total de livros: {n_books}")
print(f"- Embedding size: {EMBEDDING_SIZE}")
print(f"- Epochs treinados: {len(history.history['loss'])}")
print(f"- MAE final: {test_mae_original:.2f} (escala 0-10)")